# Qwen2.5-Coder 3B LoRA Custom Training

Upload a chat-format SFT JSONL built with `python -m training.build_sft_dataset ...`.

This notebook trains a LoRA adapter. It does not produce a final GGUF by itself.

In [ ]:
!pip install -U "transformers>=4.46" "datasets>=3" "accelerate>=1" "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.44" safetensors

In [ ]:
from google.colab import files

uploaded = files.upload()
train_path = next(iter(uploaded.keys()))
print("training file:", train_path)

In [ ]:
import json

with open(train_path, "r", encoding="utf-8") as handle:
    rows = [json.loads(line) for line in handle if line.strip()]

assert rows, "No training rows found"
for row in rows[:5]:
    assert "messages" in row, row
    assert [m["role"] for m in row["messages"]] == ["system", "user", "assistant"], row

print("records:", len(rows))
print(rows[0]["messages"][-1]["content"][:500])

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
split = dataset.train_test_split(test_size=0.1, seed=42) if len(dataset) >= 20 else {"train": dataset, "test": dataset}
split

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

base_model = "Qwen/Qwen2.5-Coder-3B-Instruct"
output_dir = "qwen25-coder-3b-sql-python-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_data = split["train"].map(format_example, remove_columns=split["train"].column_names)
eval_data = split["test"].map(format_example, remove_columns=split["test"].column_names)
print(train_data[0]["text"][:800])

In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer

sft_config_kwargs = {
    "output_dir": output_dir,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "max_length": 2048,
    "logging_steps": 5,
    "save_strategy": "epoch",
    "eval_strategy": "epoch",
    "fp16": False,
    "bf16": False,
    "optim": "paged_adamw_8bit",
    "packing": False,
    "dataset_text_field": "text",
}
sft_config_params = inspect.signature(SFTConfig).parameters
if "max_length" not in sft_config_params and "max_seq_length" in sft_config_params:
    sft_config_kwargs["max_seq_length"] = sft_config_kwargs.pop("max_length")
if "eval_strategy" not in sft_config_params and "evaluation_strategy" in sft_config_params:
    sft_config_kwargs["evaluation_strategy"] = sft_config_kwargs.pop("eval_strategy")
sft_config_kwargs = {k: v for k, v in sft_config_kwargs.items() if k in sft_config_params}

training_args = SFTConfig(**sft_config_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_data,
    "eval_dataset": eval_data,
    "peft_config": peft_config,
}
trainer_params = inspect.signature(SFTTrainer).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

trainer.train()

In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

manifest = {
    "base_model": base_model,
    "adapter_dir": output_dir,
    "record_count": len(rows),
    "method": "QLoRA",
    "target": "SQL/Python data engineering assistant",
}
with open(f"{output_dir}/training_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

!zip -r qwen25-coder-3b-sql-python-lora.zip qwen25-coder-3b-sql-python-lora
files.download("qwen25-coder-3b-sql-python-lora.zip")